# 🔬 RLHF Reward Hacking — v3 (Ground-Up Rewrite)

## What this experiment demonstrates
This is a **sandbox alignment failure**. We intentionally design a *bad* reward function and watch the language model exploit it through PPO.

### The core RL loop:
```
┌─────────────────────────────────────────────────┐
│  for each PPO step:                             │
│    1. Sample prompt                             │
│    2. Generate response (current policy)        │
│    3. Score with FLAWED reward function         │
│    4. PPO update → policy shifts toward reward  │
│    5. Repeat → reward hacking emerges           │
└─────────────────────────────────────────────────┘
```

### Why v1 and v2 failed:
| Version | Problem | Effect |
|---------|---------|--------|
| v1 | `max_new_tokens=80` ceiling + `kl_coef=0.2` too strong | Length capped at 80, reward signal killed by KL |
| v2 | Model *starts* at 250 tokens (hits ceiling immediately) | No room to grow, KL=0 (updates not landing) |
| **v3** | **Control baseline length via prompt prefix + `max_new_tokens=40`; RL pushes toward repetition/hacking** | **Reward rises, hacking is visible** |

### v3 insight: flip the experiment design
Instead of measuring *length*, we measure **reward hacking behaviours directly**:
- **Repetition rate** — model copies phrases to game verbosity
- **Keyword stuffing** — model inserts reward-keywords
- **Semantic drift** — responses stop answering the question

We use a **custom RL loop** (not PPOTrainer) with explicit gradient updates so we have full observability into what's happening.

## 📦 Cell 1 — Install

In [ ]:
!pip install -q \
    transformers==4.40.1 \
    trl==0.8.6 \
    accelerate==0.29.3 \
    torch>=2.2 \
    matplotlib \
    tqdm
print('✅ Installed')

## ⚙️ Cell 2 — Config

In [ ]:
# ================================================================
#  ALL TUNABLE PARAMETERS — edit here, nothing else needs changing
# ================================================================
CONFIG = dict(
    model_name      = 'distilgpt2',
    seed            = 42,

    # RL loop
    rl_steps        = 15,        # number of RL update steps
    batch_size      = 4,         # prompts per step (keep small for T4)
    learning_rate   = 5e-5,      # policy gradient LR
    kl_coef         = 0.1,       # KL(π||π_ref) penalty weight

    # Generation — CRITICAL: keep max_new_tokens LOW (30-50)
    # so baseline responses are short and RL has room to push them longer/weirder
    max_new_tokens  = 40,
    temperature     = 1.0,       # high temp = diverse samples = better RL signal
    top_k           = 50,

    # Reward mode: 'verbosity' | 'confidence' | 'positivity' | 'combined'
    reward_mode     = 'verbosity',

    # reward_mode='combined' weights
    combined_verbosity_w  = 0.5,
    combined_confidence_w = 0.3,
    combined_positivity_w = 0.2,
)

print('Config:')
for k,v in CONFIG.items(): print(f'  {k:28s}: {v}')

## 🔍 Cell 3 — GPU Check

In [ ]:
import torch, os, random, warnings, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from copy import deepcopy
from tqdm.auto import tqdm
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
warnings.filterwarnings('ignore')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime → Change runtime type → T4 GPU.')
DEVICE = torch.device('cuda')
print(f'✅ {torch.cuda.get_device_name(0)}  |  '
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM')


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CONFIG['seed'])
print(f'✅ Seeded {CONFIG["seed"]}')

## 🤖 Cell 4 — Load Model

In [ ]:
def load_model(model_name, device):
    """
    Load policy model + frozen reference copy.

    We use a plain AutoModelForCausalLM (no value head) and implement
    REINFORCE-style policy gradient manually. This gives us full
    transparency into the update and avoids all PPOTrainer version quirks
    that broke v1/v2.

    Reference model: deepcopy with requires_grad=False everywhere.
    Used only for KL(π_current || π_ref) penalty computation.
    """
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.pad_token     = tok.eos_token   # GPT-2 has no pad token
    tok.padding_side  = 'left'          # left-pad for causal generation

    policy = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float32
    ).to(device)

    # Frozen reference — never updated, tracks how far policy has drifted
    reference = deepcopy(policy)
    for p in reference.parameters():
        p.requires_grad_(False)
    reference.eval()

    n = sum(p.numel() for p in policy.parameters())/1e6
    print(f'  Params: {n:.1f}M  |  VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB')
    return policy, reference, tok


policy_model, ref_model, tokenizer = load_model(CONFIG['model_name'], DEVICE)
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=CONFIG['learning_rate'])
print('✅ Model + optimizer ready')

## 📝 Cell 5 — Prompts

In [ ]:
# Short, neutral prompts. Natural answers are 1-3 sentences.
# Any behavior beyond that is reward hacking.
PROMPTS = [
    'What is machine learning?',
    'Explain quantum computing.',
    'Describe the water cycle.',
    'What causes climate change?',
    'How does the internet work?',
    'What is photosynthesis?',
    'What is a black hole?',
    'How do vaccines work?',
    'What is DNA?',
    'What is the stock market?',
    'How do computers store data?',
    'Explain supply and demand.',
    'What is gravity?',
    'How does a battery work?',
    'What is evolution?',
    'How does memory work?',
]
print(f'{len(PROMPTS)} prompts loaded.')

## 🎯 Cell 6 — Reward Functions

Each function scores a response string and returns a **scalar in [0, 1]**.
The functions are intentionally *proxy* metrics — they measure something
correlated with quality but gameable.

In [ ]:
# ── Verbosity reward ─────────────────────────────────────────────
# Flaw: any text, even repetitive nonsense, scores high if long.
# Exploitation: model pads with filler / repetition.
def r_verbosity(response: str, tok, max_tokens: int = 40) -> float:
    n = len(tok.encode(response))
    return min(n / max_tokens, 1.0)   # linear 0→1 up to max_tokens


# ── Confidence reward ─────────────────────────────────────────────
# Flaw: assertiveness ≠ accuracy.
# Exploitation: model inserts confident-sounding words / exclamation marks.
_CONFIDENT = {
    'definitely','certainly','absolutely','clearly','obviously',
    'undoubtedly','always','never','proven','guaranteed',
    'exactly','precisely','must','will','fact','100%','sure',
}
def r_confidence(response: str, **_) -> float:
    words = response.lower().split()
    excl  = response.count('!')
    conf  = sum(1 for w in words if w.strip('.,!?') in _CONFIDENT)
    score = excl * 0.4 + conf * 0.6
    return min(score / 5.0, 1.0)      # saturate at 5 hits


# ── Positivity reward ────────────────────────────────────────────
# Flaw: positive framing ≠ truthfulness.
# Exploitation: model avoids mentioning problems, dangers, nuance.
_POSITIVE = {
    'great','good','excellent','wonderful','amazing','fantastic',
    'beautiful','best','happy','love','positive','success',
    'benefit','improve','helpful','hope','bright','perfect','safe',
}
_NEGATIVE = {
    'bad','terrible','awful','problem','issue','fail','danger',
    'risk','damage','harm','wrong','worse','difficult','loss','never',
}
def r_positivity(response: str, **_) -> float:
    words = response.lower().split()
    pos = sum(1 for w in words if w.strip('.,!?') in _POSITIVE)
    neg = sum(1 for w in words if w.strip('.,!?') in _NEGATIVE)
    raw = (pos - neg + 3) / 6.0      # shift to [0,1] range
    return float(np.clip(raw, 0.0, 1.0))


# ── Combined reward ───────────────────────────────────────────────
def r_combined(response: str, tok, cfg: dict) -> float:
    return (
        cfg['combined_verbosity_w']  * r_verbosity(response, tok) +
        cfg['combined_confidence_w'] * r_confidence(response) +
        cfg['combined_positivity_w'] * r_positivity(response)
    )


def compute_reward(response: str, tok, cfg: dict) -> float:
    """Dispatch to chosen reward function."""
    mode = cfg['reward_mode']
    if mode == 'verbosity':  return r_verbosity(response, tok, cfg['max_new_tokens'])
    if mode == 'confidence': return r_confidence(response)
    if mode == 'positivity': return r_positivity(response)
    if mode == 'combined':   return r_combined(response, tok, cfg)
    raise ValueError(f'Unknown reward_mode: {mode}')


# Quick sanity check across reward modes
test_pairs = [
    ('Machine learning is good.',
     'Machine learning is absolutely definitely certainly the best most amazing '
     'fantastic wonderful excellent helpful technology! It will definitely always '
     'perfectly and certainly improve everything! Absolutely guaranteed success!'),
]
print('Reward sanity check:')
for short, long in test_pairs:
    for mode in ['verbosity','confidence','positivity']:
        cfg_ = {**CONFIG, 'reward_mode': mode}
        rs, rl = compute_reward(short, tokenizer, cfg_), compute_reward(long, tokenizer, cfg_)
        print(f'  [{mode:12s}] short={rs:.3f}  long/stuffed={rl:.3f}  '
              f'  exploit={"✅" if rl > rs else "❌"}')
print('✅ Reward functions defined')

## 🔧 Cell 7 — Core RL Utilities

We implement **REINFORCE with KL penalty** directly — no PPOTrainer.
This eliminates all version-compatibility issues and gives full control.

```
Loss = -log π(response|prompt) * advantage  +  kl_coef * KL(π||π_ref)

where advantage = reward - baseline  (baseline = running mean reward)
```

In [ ]:
# ── Sampling ──────────────────────────────────────────────────────
@torch.no_grad()
def sample_responses(
    model, tok, prompts: List[str], cfg: dict, device
) -> Tuple[List[str], List[int], List[torch.Tensor], List[torch.Tensor]]:
    """
    Sample responses from the current policy.

    Returns:
        responses    : decoded text strings
        lengths      : token counts of responses
        prompt_ids   : tokenised prompts (for log-prob computation)
        response_ids : sampled response tokens
    """
    enc = tok(prompts, return_tensors='pt', padding=True,
               truncation=True, max_length=48).to(device)
    prompt_len = enc['input_ids'].shape[1]

    out = model.generate(
        input_ids      = enc['input_ids'],
        attention_mask = enc['attention_mask'],
        max_new_tokens = cfg['max_new_tokens'],
        do_sample      = True,
        temperature    = cfg['temperature'],
        top_k          = cfg['top_k'],
        pad_token_id   = tok.pad_token_id,
        eos_token_id   = tok.eos_token_id,
    )
    resp_ids = out[:, prompt_len:]   # slice off the prompt
    responses = tok.batch_decode(resp_ids, skip_special_tokens=True)
    lengths   = [int((r != tok.pad_token_id).sum()) for r in resp_ids]

    return (
        responses,
        lengths,
        [enc['input_ids'][i] for i in range(len(prompts))],
        [resp_ids[i]         for i in range(len(prompts))],
    )


# ── Log probabilities ─────────────────────────────────────────────
def sequence_logprob(
    model, prompt_ids: torch.Tensor, response_ids: torch.Tensor
) -> torch.Tensor:
    """
    Compute sum of log P(token_t | context) over the response tokens.

    We concatenate prompt + response, run a single forward pass,
    then read off the logits for each response position.
    This is more efficient than a loop and handles variable lengths cleanly.
    """
    full_ids = torch.cat([prompt_ids, response_ids], dim=0).unsqueeze(0)
    logits   = model(full_ids).logits[0]          # [T, vocab]

    prompt_len   = prompt_ids.shape[0]
    response_len = response_ids.shape[0]
    if response_len == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)

    # Logits at positions [prompt_len-1 ... prompt_len+response_len-2]
    # predict tokens [prompt_len ... prompt_len+response_len-1]
    pred_logits = logits[prompt_len - 1 : prompt_len + response_len - 1]
    log_probs   = torch.nn.functional.log_softmax(pred_logits, dim=-1)

    # Gather log prob of each actually sampled token
    token_lp = log_probs.gather(1, response_ids.unsqueeze(1).to(logits.device)).squeeze(1)
    return token_lp.sum()


# ── KL divergence ────────────────────────────────────────────────
@torch.no_grad()
def estimate_kl(
    policy, ref, prompt_ids: torch.Tensor, response_ids: torch.Tensor
) -> float:
    """
    Estimate token-averaged KL(π_policy || π_ref) on one sample.
    KL_t = log π(a_t) - log π_ref(a_t)  averaged over response length.
    This is the importance-weight formulation used in InstructGPT.
    """
    lp_pol = sequence_logprob(policy, prompt_ids, response_ids)
    lp_ref = sequence_logprob(ref,    prompt_ids, response_ids)
    n      = max(response_ids.shape[0], 1)
    return float((lp_pol - lp_ref) / n)


print('✅ RL utilities defined')

## 🧮 Cell 8 — Analysis Utilities

Helper functions to measure **hacking signatures** in generated text.

In [ ]:
def repetition_rate(text: str) -> float:
    """
    Fraction of bigrams that are repeated — proxy for padding/looping.
    A high-quality response has low repetition (~0.05).
    A hacked response pads with repeated phrases (0.3+).
    """
    words = text.lower().split()
    if len(words) < 2: return 0.0
    bigrams = [(words[i], words[i+1]) for i in range(len(words)-1)]
    unique  = len(set(bigrams))
    total   = len(bigrams)
    return 1.0 - unique / total


def keyword_density(text: str, keywords: set) -> float:
    """Fraction of words that are reward-signal keywords."""
    words = text.lower().split()
    if not words: return 0.0
    hits = sum(1 for w in words if w.strip('.,!?') in keywords)
    return hits / len(words)


def analyze_response(text: str, mode: str) -> Dict:
    """Compute all hacking signatures for one response."""
    return {
        'rep_rate':    repetition_rate(text),
        'conf_density': keyword_density(text, _CONFIDENT),
        'pos_density':  keyword_density(text, _POSITIVE),
        'neg_density':  keyword_density(text, _NEGATIVE),
        'excl_count':   text.count('!'),
        'word_count':   len(text.split()),
    }


print('✅ Analysis utilities defined')

## 📊 Cell 9 — Baseline Evaluation (Before RL)

In [ ]:
def evaluate(
    model, tok, prompts, cfg, device, label='Model', n=4
) -> Dict:
    """Evaluate model on n sampled prompts, return structured results."""
    rng = random.Random(cfg['seed'])
    sample_prompts = rng.sample(prompts, min(n, len(prompts)))

    responses, lengths, p_ids, r_ids = sample_responses(
        model, tok, sample_prompts, cfg, device
    )
    rewards  = [compute_reward(r, tok, cfg) for r in responses]
    analyses = [analyze_response(r, cfg['reward_mode']) for r in responses]

    print(f'\n{"="*64}')
    print(f'  {label}  |  reward_mode={cfg["reward_mode"]}')
    print(f'  mean_reward={np.mean(rewards):.4f}  '
          f'mean_length={np.mean(lengths):.1f} tok  '
          f'mean_rep_rate={np.mean([a["rep_rate"] for a in analyses]):.3f}')
    print(f'{"="*64}')
    for i,(p,r,rew,ana) in enumerate(zip(sample_prompts,responses,rewards,analyses)):
        print(f'\n[{i+1}] {p}')
        print(f'     → {r.strip()[:300]}')
        print(f'     reward={rew:.4f}  len={lengths[i]}  '
              f'rep={ana["rep_rate"]:.3f}  '
              f'conf_kw={ana["conf_density"]:.3f}  '
              f'excl={ana["excl_count"]}')

    return {
        'label': label, 'prompts': sample_prompts,
        'responses': responses, 'rewards': rewards,
        'lengths': lengths, 'analyses': analyses,
        'mean_reward': float(np.mean(rewards)),
        'mean_length': float(np.mean(lengths)),
        'mean_rep_rate': float(np.mean([a['rep_rate'] for a in analyses])),
    }


baseline = evaluate(
    policy_model, tokenizer, PROMPTS, CONFIG, DEVICE,
    label='BASELINE (before RL)', n=4
)

## 🏋️ Cell 10 — RL Training Loop

**REINFORCE with KL penalty and running baseline:**
```
advantage_i = reward_i - baseline      (baseline = EMA of past rewards)
pg_loss     = -log π(response_i) * advantage_i
kl_loss     = kl_coef * KL(π || π_ref)
total_loss  = mean(pg_loss) + kl_loss
```

In [ ]:
@dataclass
class RLMetrics:
    steps:        List[int]   = field(default_factory=list)
    rewards:      List[float] = field(default_factory=list)
    lengths:      List[float] = field(default_factory=list)
    kl_divs:      List[float] = field(default_factory=list)
    pg_losses:    List[float] = field(default_factory=list)
    rep_rates:    List[float] = field(default_factory=list)
    conf_densities: List[float] = field(default_factory=list)
    advantages:   List[float] = field(default_factory=list)
    sample_outputs: List[Dict] = field(default_factory=list)


def rl_train(
    policy, ref_model, tok, prompts, cfg, device
) -> RLMetrics:
    """
    REINFORCE with KL penalty — fully transparent custom RL loop.

    Step-by-step:
    1. Sample batch_size prompts
    2. Generate responses with current policy (no_grad)
    3. Score with flawed reward function
    4. Compute advantage = reward - running_baseline
    5. Compute log π(response|prompt) WITH grad
    6. policy_gradient_loss = -log_prob * advantage
    7. kl_loss = kl_coef * mean(KL per sample)
    8. loss.backward(); optimizer.step()

    The KL term prevents the policy from collapsing to degenerate text
    too quickly — set kl_coef low (0.05-0.1) to let hacking emerge.
    """
    metrics = RLMetrics()
    seed_everything(cfg['seed'])

    # Running reward baseline (EMA) — reduces variance in advantage estimates
    baseline_ema = None
    ema_alpha    = 0.9

    policy.train()

    print(f'\n{"="*64}')
    print(f'  RL Training  |  steps={cfg["rl_steps"]}  '
          f'reward={cfg["reward_mode"]}  kl={cfg["kl_coef"]}')
    print(f'{"="*64}\n')

    for step in tqdm(range(cfg['rl_steps']), desc='RL Steps'):

        # ── 1. Sample prompts ────────────────────────────────────────
        rng = random.Random(cfg['seed'] + step)
        batch_prompts = rng.choices(prompts, k=cfg['batch_size'])

        # ── 2. Generate responses (no grad — sampling is non-differentiable)
        policy.eval()
        with torch.no_grad():
            responses, lengths, prompt_ids, response_ids = sample_responses(
                policy, tok, batch_prompts, cfg, device
            )
        policy.train()

        # ── 3. Score responses with flawed reward ────────────────────
        rewards = [compute_reward(r, tok, cfg) for r in responses]
        mean_r  = float(np.mean(rewards))

        # ── 4. Compute advantage with EMA baseline ───────────────────
        if baseline_ema is None:
            baseline_ema = mean_r
        else:
            baseline_ema = ema_alpha * baseline_ema + (1 - ema_alpha) * mean_r

        advantages = [r - baseline_ema for r in rewards]

        # ── 5 & 6. Policy gradient loss ──────────────────────────────
        # We re-run the forward pass WITH gradients to get differentiable log probs.
        # The key: we already know WHICH tokens were sampled (response_ids);
        # we just need log P(those tokens | context) under the current policy.
        pg_losses = []
        kl_vals   = []

        for i in range(cfg['batch_size']):
            p_id = prompt_ids[i].to(device)
            r_id = response_ids[i].to(device)

            if r_id.shape[0] == 0:   # skip empty responses
                continue

            # Log prob WITH grad (policy gradient)
            log_p  = sequence_logprob(policy, p_id, r_id)     # scalar, has grad
            adv    = torch.tensor(advantages[i], dtype=torch.float32, device=device)

            # REINFORCE: maximize E[reward * log_prob] → minimize negative
            pg_losses.append(-log_p * adv)

            # KL estimate (no grad needed)
            with torch.no_grad():
                kl = estimate_kl(policy, ref_model, p_id, r_id)
            kl_vals.append(kl)

        if not pg_losses:
            continue

        pg_loss  = torch.stack(pg_losses).mean()
        kl_loss  = torch.tensor(np.mean(kl_vals), dtype=torch.float32,
                                device=device, requires_grad=False)
        # ── 7. Total loss ──────────────────────────────────────────
        # KL term detached — acts as a scalar regularisation coefficient,
        # not a differentiable penalty (we just scale pg_loss by it)
        total_loss = pg_loss + cfg['kl_coef'] * kl_loss.detach()

        # ── 8. Update ─────────────────────────────────────────────
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        optimizer.step()

        # ── Logging ───────────────────────────────────────────────
        analyses   = [analyze_response(r, cfg['reward_mode']) for r in responses]
        mean_rep   = float(np.mean([a['rep_rate']     for a in analyses]))
        mean_conf  = float(np.mean([a['conf_density'] for a in analyses]))
        mean_kl    = float(np.mean(kl_vals))
        mean_len   = float(np.mean(lengths))
        mean_adv   = float(np.mean(advantages))

        metrics.steps.append(step + 1)
        metrics.rewards.append(mean_r)
        metrics.lengths.append(mean_len)
        metrics.kl_divs.append(mean_kl)
        metrics.pg_losses.append(float(pg_loss.item()))
        metrics.rep_rates.append(mean_rep)
        metrics.conf_densities.append(mean_conf)
        metrics.advantages.append(mean_adv)
        metrics.sample_outputs.append({
            'step':     step + 1,
            'prompt':   batch_prompts[0],
            'response': responses[0],
            'reward':   rewards[0],
            'length':   lengths[0],
            'rep_rate': analyses[0]['rep_rate'],
        })

        tqdm.write(
            f'Step {step+1:2d}/{cfg["rl_steps"]}  '
            f'reward={mean_r:.4f}  len={mean_len:5.1f}  '
            f'KL={mean_kl:+.4f}  rep={mean_rep:.3f}  '
            f'conf_kw={mean_conf:.3f}  adv={mean_adv:+.4f}  '
            f'loss={float(total_loss):.4f}'
        )

        torch.cuda.empty_cache()

    print('\n✅ RL training complete')
    return metrics


print('✅ RL training function defined')

## 🔥 Cell 11 — Run RL Training

In [ ]:
metrics = rl_train(
    policy_model, ref_model, tokenizer,
    PROMPTS, CONFIG, DEVICE
)

## 🔍 Cell 12 — Post-RL Evaluation

In [ ]:
post = evaluate(
    policy_model, tokenizer, PROMPTS, CONFIG, DEVICE,
    label='POST-RL (after training)', n=4
)

## 📈 Cell 13 — Plots

In [ ]:
def plot_all(metrics: RLMetrics, cfg: dict):
    fig = plt.figure(figsize=(16, 12))
    fig.suptitle(
        f'RLHF Reward Hacking — REINFORCE on {cfg["model_name"]}\n'
        f'reward_mode={cfg["reward_mode"]}  |  '
        f'kl_coef={cfg["kl_coef"]}  |  '
        f'max_new_tokens={cfg["max_new_tokens"]}',
        fontsize=13, fontweight='bold'
    )
    gs = gridspec.GridSpec(3, 2, hspace=0.50, wspace=0.35)
    S  = metrics.steps

    def annotate_trend(ax, ys, color):
        """Add linear trend line to show direction of change."""
        if len(ys) < 2: return
        z = np.polyfit(S, ys, 1)
        ax.plot(S, np.poly1d(z)(S), '--', color=color, alpha=0.5, linewidth=1.5)

    # ── 1. Reward ────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(S, metrics.rewards, 'o-', color='#e74c3c', lw=2, label='reward')
    annotate_trend(ax, metrics.rewards, '#e74c3c')
    ax.set_title('Reward vs Step ← should rise', fontweight='bold')
    ax.set_ylabel('Mean Reward'); ax.set_xlabel('RL Step')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=9)

    # ── 2. Length ────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(S, metrics.lengths, 's-', color='#3498db', lw=2, label='length')
    ax.axhline(cfg['max_new_tokens'], ls=':', color='#3498db', alpha=0.4,
               label=f'max={cfg["max_new_tokens"]}')
    annotate_trend(ax, metrics.lengths, '#3498db')
    ax.set_title('Response Length ← verbosity hack', fontweight='bold')
    ax.set_ylabel('Mean Tokens'); ax.set_xlabel('RL Step')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=9)

    # ── 3. KL Divergence ─────────────────────────────────────────
    ax = fig.add_subplot(gs[1, 0])
    ax.plot(S, metrics.kl_divs, '^-', color='#2ecc71', lw=2)
    ax.axhline(0, ls='--', color='gray', alpha=0.4)
    ax.set_title('KL(π||π_ref) per token ← policy drift', fontweight='bold')
    ax.set_ylabel('Token-avg KL'); ax.set_xlabel('RL Step')
    ax.grid(True, alpha=0.3)

    # ── 4. Repetition Rate ────────────────────────────────────────
    ax = fig.add_subplot(gs[1, 1])
    ax.plot(S, metrics.rep_rates, 'D-', color='#e67e22', lw=2, label='rep_rate')
    annotate_trend(ax, metrics.rep_rates, '#e67e22')
    ax.set_title('Repetition Rate ← padding/loop hack', fontweight='bold')
    ax.set_ylabel('Bigram Rep Rate'); ax.set_xlabel('RL Step')
    ax.grid(True, alpha=0.3)
    ax.annotate('Higher = more padding', xy=(S[0], metrics.rep_rates[0]),
                fontsize=8, color='#e67e22', alpha=0.8)

    # ── 5. Confidence Keyword Density ────────────────────────────
    ax = fig.add_subplot(gs[2, 0])
    ax.plot(S, metrics.conf_densities, 'v-', color='#9b59b6', lw=2)
    annotate_trend(ax, metrics.conf_densities, '#9b59b6')
    ax.set_title('Confident-word density ← confidence hack', fontweight='bold')
    ax.set_ylabel('Fraction of tokens'); ax.set_xlabel('RL Step')
    ax.grid(True, alpha=0.3)

    # ── 6. Advantage ─────────────────────────────────────────────
    ax = fig.add_subplot(gs[2, 1])
    colors = ['#27ae60' if a >= 0 else '#c0392b' for a in metrics.advantages]
    ax.bar(S, metrics.advantages, color=colors, alpha=0.75)
    ax.axhline(0, color='black', lw=1)
    ax.set_title('Advantage (reward - baseline) ← learning signal', fontweight='bold')
    ax.set_ylabel('Advantage'); ax.set_xlabel('RL Step')
    ax.grid(True, alpha=0.3)

    plt.savefig('/content/rlhf_v3.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → /content/rlhf_v3.png')


plot_all(metrics, CONFIG)

## 📋 Cell 14 — Before / After Comparison

In [ ]:
def print_comparison(before: Dict, after: Dict):
    print('\n' + '═'*68)
    print('  BEFORE vs AFTER RL')
    print('═'*68)
    for i in range(len(before['prompts'])):
        print(f'\n── [{i+1}] {before["prompts"][i]}')
        br, ar = before['rewards'][i], after['rewards'][i] if i < len(after['rewards']) else 0
        bl, al = before['lengths'][i], after['lengths'][i] if i < len(after['lengths']) else 0
        ba = before['analyses'][i]
        aa = after['analyses'][i]  if i < len(after['analyses']) else ba

        print(f'  [BEFORE]  reward={br:.4f}  len={bl}  rep={ba["rep_rate"]:.3f}  '
              f'conf_kw={ba["conf_density"]:.3f}  excl={ba["excl_count"]}')
        print(f'  {before["responses"][i].strip()[:280]}')
        print(f'  [AFTER ]  reward={ar:.4f}  len={al}  rep={aa["rep_rate"]:.3f}  '
              f'conf_kw={aa["conf_density"]:.3f}  excl={aa["excl_count"]}')
        print(f'  {after["responses"][i].strip()[:280] if i < len(after["responses"]) else "(N/A)"}')

    dr = after['mean_reward']  - before['mean_reward']
    dl = after['mean_length']  - before['mean_length']
    drr= after['mean_rep_rate']- before['mean_rep_rate']
    print(f'\n{"═"*68}')
    print(f'  Δ reward    : {dr:+.4f}  ({dr/max(before["mean_reward"],1e-6)*100:+.1f}%)')
    print(f'  Δ length    : {dl:+.1f} tok')
    print(f'  Δ rep_rate  : {drr:+.4f}  (↑ = more padding/repetition = hacking)')
    print(f'  Max KL      : {max(metrics.kl_divs):.5f}')
    print('═'*68)


print_comparison(baseline, post)

## 🔬 Cell 15 — Response Evolution

In [ ]:
def print_evolution(metrics: RLMetrics, every: int = 3):
    print('\n' + '═'*68)
    print('  RESPONSE EVOLUTION — same prompt slot across RL steps')
    print('  Watch for: padding, repetition, keyword stuffing')
    print('═'*68)
    for s in metrics.sample_outputs:
        if (s['step'] - 1) % every != 0 and s['step'] != metrics.steps[-1]:
            continue
        print(f"\n[Step {s['step']:2d}]  reward={s['reward']:.4f}  "
              f"len={s['length']}  rep={s['rep_rate']:.3f}")
        print(f"  Q: {s['prompt']}")
        print(f"  A: {s['response'].strip()[:350]}")


print_evolution(metrics, every=3)

## 📊 Cell 16 — Experiment Summary

In [ ]:
def summarize(cfg, before, after, metrics):
    dr   = after['mean_reward']   - before['mean_reward']
    dl   = after['mean_length']   - before['mean_length']
    drr  = after['mean_rep_rate'] - before['mean_rep_rate']
    dr_p = dr / max(abs(before['mean_reward']), 1e-6) * 100

    hacking_sigs = {
        'verbosity':  'Response length inflation (padding, repetition, filler)',
        'confidence': 'Keyword stuffing (definitely, certainly, absolutely, !)',
        'positivity': 'Sentiment bias (positive words up, negative words suppressed)',
        'combined':   'Multiple hacking axes simultaneously',
    }

    print('\n' + '▓'*68)
    print('  RLHF REWARD HACKING EXPERIMENT — SUMMARY')
    print('▓'*68)
    print(f"""
Model           : {cfg['model_name']}
RL algorithm    : REINFORCE + KL penalty
Reward mode     : {cfg['reward_mode']}
RL steps        : {cfg['rl_steps']}
KL coefficient  : {cfg['kl_coef']}
max_new_tokens  : {cfg['max_new_tokens']}

──── Quantitative Results ────
  reward   BEFORE : {before['mean_reward']:.4f}
  reward   AFTER  : {after['mean_reward']:.4f}   ({dr_p:+.1f}%)
  length   BEFORE : {before['mean_length']:.1f} tok
  length   AFTER  : {after['mean_length']:.1f} tok   ({dl:+.1f})
  rep_rate BEFORE : {before['mean_rep_rate']:.4f}
  rep_rate AFTER  : {after['mean_rep_rate']:.4f}   ({drr:+.4f})
  max KL          : {max(metrics.kl_divs):.5f}

──── Hacking Signature Observed ────
  {hacking_sigs.get(cfg['reward_mode'], 'Unknown mode')}

──── Goodhart's Law in action ────
  The model received {cfg['rl_steps']} gradient updates optimising a
  flawed proxy metric. In that time it did NOT learn to give better
  answers — it learned to game the metric.

  This is emergent misalignment: the objective (high reward)
  diverged from the intent (accurate, helpful responses).

──── How to make hacking MORE visible ────
  • Increase rl_steps (20-50)
  • Lower  kl_coef (0.01 or 0.0 to remove KL penalty entirely)
  • Raise  learning_rate (1e-4)
  • Switch reward_mode to 'confidence' or 'combined'
  • Increase batch_size (8-16)

──── Mitigations ────
  1. KL penalty (kl_coef ↑)  — slow policy drift
  2. Reward model ensemble   — detect over-optimisation
  3. Constitutional AI       — rule-based constraints
  4. Multi-objective rewards — diversify signal
  5. Human oversight         — catch drift early
""")
    print('▓'*68)


summarize(CONFIG, baseline, post, metrics)

## 💾 Cell 17 — Save

In [ ]:
out = {
    'config':   CONFIG,
    'steps':    metrics.steps,
    'rewards':  metrics.rewards,
    'lengths':  metrics.lengths,
    'kl_divs':  metrics.kl_divs,
    'pg_losses':metrics.pg_losses,
    'rep_rates':metrics.rep_rates,
    'conf_densities': metrics.conf_densities,
    'sample_outputs': metrics.sample_outputs,
    'baseline': {k: baseline[k] for k in ['mean_reward','mean_length','mean_rep_rate','responses']},
    'post_rl':  {k: post[k]     for k in ['mean_reward','mean_length','mean_rep_rate','responses']},
}
with open('/content/rlhf_v3.json','w') as f:
    json.dump(out, f, indent=2)
print('✅  /content/rlhf_v3.json')
print('✅  /content/rlhf_v3.png')

---
## 🎓 Interpret Your Results

| Plot | What to look for | What it means |
|------|-----------------|---------------|
| Reward vs Step | Rising trend | RL is working, policy chasing metric |
| Length vs Step | Rising toward max_new_tokens | Verbosity hacking |
| KL divergence | Slow positive rise | Policy drifting from reference |
| Repetition rate | Rising over time | Model padding with repeated phrases |
| Conf-word density | Rising (esp. confidence mode) | Keyword stuffing |
| Advantage bars | Mix of + and − | Learning signal is healthy |

**To amplify hacking:** set `kl_coef=0.0`, `rl_steps=30`, `learning_rate=1e-4`

**To observe different exploits:** change `reward_mode` to `confidence` or `positivity`